In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

# https://stackoverflow.com/questions/21971449/how-do-i-increase-the-cell-width-of-the-jupyter-ipython-notebook-in-my-browser
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

from matplotlib import pyplot as plt
import numpy as np
from pathlib import Path
import json
import pickle
import random
from IPython.display import display, Image, clear_output
from PIL import Image as PILImage
# from pigeonXT import annotate
from collections import defaultdict
import sys
import os
from glob import glob
from pprint import pprint
import pandas as pd
import utils
import faiss
import networkx as nx
import textwrap
import ipyplot
from itertools import zip_longest          # lets us iterate “row-wise”
from IPython.display import HTML
import html
from interactive_annotation import build_annotation_ui

In [ ]:
def show_imgs_sidebyside(imgs, captions, grayscale_imgs=None, title=None, figsize=None):
    """ Assume images already loaded """
    NUM_ROWS = 1
    IMGS_PER_ROW = len(imgs)
    f, ax = plt.subplots(NUM_ROWS, IMGS_PER_ROW, figsize=figsize if figsize else (30,4))
    
    for i, (img, caption) in enumerate(zip(imgs, captions)):
        ax[i].imshow(img, cmap='gray')
        # Add pad to increase the distance between the image and the title
        if isinstance(caption, str):
            ax[i].set_title(caption, pad=20)
        else:
            # If the caption is a dict; add pad manually to the title text:
            ax[i].set_title(caption.get('text', ''), pad=20, **{k: v for k, v in caption.items() if k != 'text'})
        ax[i].axes.xaxis.set_ticks([])
        ax[i].axes.yaxis.set_ticks([])
        if title is not None and i == 0:            
            ax[i].set_ylabel(**title, rotation=0, fontsize=14, labelpad=22)
    plt.tight_layout()
        
    if grayscale_imgs is not None:
        assert len(grayscale_imgs) == len(imgs)
        f1, ax1 = plt.subplots(NUM_ROWS, IMGS_PER_ROW, figsize=(16,6))
        for i, img in enumerate(grayscale_imgs):
            ax1[i].imshow(img)
            ax1[i].axes.xaxis.set_ticks([])
            ax1[i].axes.yaxis.set_ticks([])
            if title is not None and i == 0:
                ax1[i].set_ylabel(**title, rotation=0, fontsize=10, labelpad=22)
        plt.tight_layout()
    plt.show()


In [ ]:


def generate_individual_images_side_by_side(img_paths, captions, grayscale_imgs=None, output_dir="./ipynb_plot_output/", title=None, figsize=None):
    """
    Generates individual image files from a list of images and captions,
    conceptually arranged in a row.
    Args:
        img_paths (list): A list of paths.
        captions (list): A list of captions (strings or dictionaries for title properties).
        grayscale_imgs (list, optional): A list of grayscale image data. Defaults to None.
        base_filename (str, optional): The base filename for the generated images.
            Each image will be saved as f"{output_dir}/{base_filename}_{index}.png".
            Defaults to "image".
        output_dir (str, optional): The directory to save the images. Defaults to "./".
        title (dict, optional): A dictionary of title properties (applied to the conceptual "first" image). Defaults to None.
        figsize (tuple, optional): The figure size for each individual image. Defaults to None.
    """
    os.makedirs(output_dir, exist_ok=True)
    imgs = [np.array(PILImage.open(p)) for p in img_paths]
    output_paths = []

    for i, (img, caption) in enumerate(zip(imgs, captions)):
        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(img, cmap='gray')
        if isinstance(caption, str):
            wrapped_caption = textwrap.fill(caption, width=30)
            ax.set_title(wrapped_caption)
        else:
            # If the caption is a dict, assume the title text is under the key 'text'
            if 'text' in caption:
                caption['text'] = textwrap.fill(caption['text'], width=30)
            ax.set_title(**caption)
        ax.axis('off')  # Remove axes ticks and labels
        if title is not None and i == 0:
            ax.set_ylabel(**title, rotation=0, fontsize=8, labelpad=2)
        plt.tight_layout(pad=0)
        output_path = f"{output_dir}/{Path(img_paths[i]).with_suffix('').name}.png"
        plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
        output_paths.append(output_path)
        plt.close(fig)
    ipyplot.plot_images(output_paths, max_images=len(output_paths), img_width=100, show_url=False, zoom_scale=3.0)



In [ ]:
def load_matching_run(exp_dir: str, exp_prefix: str):
    # glob for everything after prefix
    print("globbing", f"{os.path.join(exp_dir, exp_prefix)}*.csv")
    files = sorted(glob(f"{os.path.join(exp_dir, exp_prefix)}*.csv"))
    print(files)
    print(f'Found {len(files)} matching files. Characters: ')
    print(', '.join([fname[-5] for fname in files]))
    data = dict()
    for fname in files:
        queries = []
        candidates = []
        sims = []
        row_paths = []
        with open(fname) as f:
            for line in f.readlines():
                line = line.strip()
                this_query = line.split('|')[0]
                # count the number of '|' in the line
                num_sep = line.count('|')
                nc = num_sep // 2
                this_candidates = line.split('|')[1:nc + 1]
                this_sims = [float(s) for s in line.split('|')[nc+1:]]
                queries.append(this_query)
                candidates.append(this_candidates)
                sims.append(this_sims)
                row_paths.append([this_query] + this_candidates)
        data[fname[-5]] = {'row_paths': row_paths, 'queries': queries, 'candidates': candidates, 'sims': sims}
        # with open(fname, 'rb') as f:
        #     data[fname[-5]] = pickle.load(f)
    return data

In [ ]:
def get_image_titles(queries, candidates, sims):
    titles = []
    for i, p in enumerate([queries] + candidates):
        book_name = utils.get_book_name(Path(p).name)
        # text wrap book name
        printer = utils.get_printer_name(Path(p).name)
        book_name = textwrap.fill(book_name, width=30)
        page_num = utils.get_page_num(Path(p).name)
        line_num = utils.get_line_num(Path(p).name)
        char_loc = utils.get_char_loc(Path(p).name)
        if i == 0:  # query
            titles.append(f"Q{i}\n{printer}\n{book_name}\npage {page_num}, line {line_num}, loc {char_loc}" + "\n(Query)")
        else:  # candidate
            titles.append(f"M{i}\n{printer}\n{book_name}\npage {page_num}, line {line_num}, loc {char_loc}" + f"\n({sims[i-1]:.2f})")
    return titles
        


In [ ]:
import html, base64, io
from pathlib import Path
from PIL import Image
import utils                                  # your helper module

# ──────────────────────────────────────────────────────────────────
# 1.  Low-level helper: build a thumbnail that fits inside
#     (max_w, max_h) and emit an <img> whose *display* size
#     is fixed to exactly that rectangle (CSS object-fit keeps AR).
# ──────────────────────────────────────────────────────────────────
def _img_tag(path, *, max_w, max_h, resample=Image.Resampling.LANCZOS):
    try:
        img = Image.open(path)
        img.thumbnail((max_w, max_h), resample)

        buf = io.BytesIO()
        img.save(buf, format='PNG')
        b64 = base64.b64encode(buf.getvalue()).decode()

        style = (
            f'width:{max_w}px; height:{max_h}px; '
            'object-fit:contain; display:block; margin:auto;'
        )
        return f'<img src="data:image/png;base64,{b64}" style="{style}">'
    except Exception as exc:
        return f'<em>Preview error: {html.escape(str(exc))}</em>'


# ──────────────────────────────────────────────────────────────────
# 2.  Main builder
# ──────────────────────────────────────────────────────────────────
def paths_to_html_table_with_details(
        paths,
        *,
        cell_width='30ch',
        table_attrs='border="1" cellpadding="4"') -> str:
    """
    HTML table where column-0 holds labels; remaining columns show
    • file name • printer • book • page • line • char_loc • preview •
    All previews are rendered in identical (max_w × max_h) boxes.
    """
    n = len(paths)

    # ---------- header row --------------------------------------------------
    header_cells = ''.join(
        f'<th style="width:{cell_width}; text-align:center;">{i}</th>'
        for i in range(n)
    )
    header_row = f'<tr><th></th>{header_cells}</tr>'

    # ---------- gather textual data ----------------------------------------
    file_names = [html.escape(Path(p).name)                         for p in paths]
    printers   = [html.escape(utils.get_printer_name(Path(p).name)) for p in paths]
    book_names = [html.escape(utils.get_book_name(Path(p).name))    for p in paths]
    page_nums  = [html.escape(str(utils.get_page_num(Path(p).name))) for p in paths]
    line_nums  = [html.escape(str(utils.get_line_num(Path(p).name))) for p in paths]
    char_locs  = [html.escape(str(utils.get_char_loc(Path(p).name))) for p in paths]

    # ---------- find the LARGEST w×h across all images ----------------------
    max_w = max_h = 0
    for p in paths:
        try:
            with Image.open(p) as im:
                w, h = im.size
            max_w = max(max_w, w)
            max_h = max(max_h, h)
        except Exception:
            # keep going; the _img_tag fallback will show an error note
            pass
    if max_w == 0 or max_h == 0:        # all failed?
        max_w = max_h = 150             # sensible default

    # ---------- build thumbnails using that common box ---------------------
    thumbnails = [_img_tag(p, max_w=max_w, max_h=max_h) for p in paths]

    # ---------- label → cells mapping --------------------------------------
    rows = [
        ("File name",  file_names),
        ("Printer",    printers),
        ("Book",       book_names),
        ("Page #",     page_nums),
        ("Line #",     line_nums),
        ("Char loc",   char_locs),
        ("Image",      thumbnails),
    ]

    # ---------- HTML row builder -------------------------------------------
    def make_row(label, cells, extra_css='text-align:center;'):
        base_css = (
            f'width:{cell_width}; max-width:{cell_width}; '
            'overflow-wrap:anywhere; word-break:break-all; '
        )
        cell_html = ''.join(
            f'<td style="{base_css}{extra_css}">{c}</td>' for c in cells
        )
        return f'<tr><th scope="row">{label}</th>{cell_html}</tr>'

    data_rows = '\n'.join(make_row(lbl, cells) for lbl, cells in rows)

    # ---------- assemble full table ----------------------------------------
    return (
        f'<table {table_attrs}>\n'
        f'{header_row}\n'
        f'{data_rows}\n'
        f'</table>'
    )

# ---------------------------------------------------------------------------
# EXAMPLE
# ---------------------------------------------------------------------------
paths = [
    '/graft2/code/nvog/git/matching/char_images3/char_H_uc/'
    'reveringham_R30863_ctbtcml_2_fortysermons1685-012_page1rline8_char72_H_uc.tif',
    '/graft2/code/nvog/git/matching/char_images3/char_H_uc/'
    'reveringham_R30863_ctbtcml_2_fortysermons1685-056_page1rline37_char4_H_uc.tif',
    # … more paths …
]


print(paths_to_html_table_with_details(paths, cell_width='15ch'))

HTML(paths_to_html_table_with_details(paths, cell_width='15ch'))

In [ ]:
import html
import base64
import io
import json
import uuid
from pathlib import Path

from PIL import Image
from IPython.display import display, HTML

# # This is a mock 'utils' module for demonstration.
# # In your actual code, you would use your own 'utils.py' file.
# class MockUtils:
#     def get_printer_name(self, p): return "mock_printer"
#     def get_book_name(self, p): return "mock_book"
#     def get_page_num(self, p): return "1"
#     def get_line_num(self, p): return "1"
#     def get_char_loc(self, p): return "1"
# utils = MockUtils()


# A global registry to hold references to our table instances
_interactive_tables = {}

# ──────────────────────────────────────────────────────────────────
# 1. Low-level helper (Unchanged)
# ──────────────────────────────────────────────────────────────────
def _img_tag(path, *, max_w, max_h, resample=Image.Resampling.LANCZOS):
    try:
        with Image.open(path) as img:
            img.thumbnail((max_w, max_h), resample)
            buf = io.BytesIO()
            img.save(buf, format='PNG')
            b64 = base64.b64encode(buf.getvalue()).decode()
            style = (
                f'width:{max_w}px; height:{max_h}px; '
                'object-fit:contain; display:block; margin:auto;'
            )
            return f'<img src="data:image/png;base64,{b64}" style="{style}">'
    except Exception as exc:
        # In a real scenario, you might have placeholder images or just this error text.
        # For this example, we return a simple error string.
        return f'<em>Preview error: {html.escape(str(exc))}</em>'


# ──────────────────────────────────────────────────────────────────
# 2. Corrected Class for Interactive Annotation (Jupyter version)
# ──────────────────────────────────────────────────────────────────
class InteractiveAnnotationTable:
    """
    Generates and displays an interactive HTML table in a local Jupyter notebook
    for data annotation. Users can select columns and export their selections.
    """
    def __init__(self, paths, output_filename="annotations.json"):
        if not paths:
            print("Warning: The 'paths' list is empty. No table will be generated.")
            return

        self.paths = paths
        self.output_filename = output_filename
        self.table_id = f"annotation-table-{str(uuid.uuid4())}"
        self.instance_id = str(uuid.uuid4())
        
        _interactive_tables[self.instance_id] = self

    def _handle_js_export(self, selected_indices_json):
        """
        Public method to save annotations. This is called by the JS in the frontend.
        """
        data = json.loads(selected_indices_json)
        selected_indices = data.get("selected_indices", [])
        selected_paths = [self.paths[i] for i in selected_indices]

        with open(self.output_filename, 'w') as f:
            json.dump({
                "selected_paths": selected_paths,
                "selected_indices": selected_indices
            }, f, indent=4)
        
        display(HTML(f"<p>✅ <b>Annotations saved!</b> {len(selected_paths)} items exported to '{self.output_filename}'</p>"))

    def _generate_html(self, cell_width='18ch', table_attrs='border="1" cellpadding="4"'):
        """
        Generates the complete HTML string, including CSS and JavaScript
        for interactivity.
        """
        n = len(self.paths)
        header_cells = ''.join(
            f'<th style="width:{cell_width}; text-align:center; cursor:pointer;" '
            f'onclick="toggleColumn(\'{self.table_id}\', {i})">{i}</th>'
            for i in range(n)
        )
        header_row = f'<tr><th style="cursor:default;"></th>{header_cells}</tr>'

        file_names = [html.escape(Path(p).name) for p in self.paths]
        printers = [html.escape(utils.get_printer_name(Path(p).name)) for p in self.paths]
        book_names = [html.escape(utils.get_book_name(Path(p).name)) for p in self.paths]
        page_nums = [html.escape(str(utils.get_page_num(Path(p).name))) for p in self.paths]
        line_nums = [html.escape(str(utils.get_line_num(Path(p).name))) for p in self.paths]
        char_locs = [html.escape(str(utils.get_char_loc(Path(p).name))) for p in self.paths]

        max_w = max_h = 0
        for p in self.paths:
            try:
                with Image.open(p) as im:
                    w, h = im.size
                    max_w = max(max_w, w)
                    max_h = max(max_h, h)
            except Exception:
                pass
        if max_w == 0 or max_h == 0: max_w, max_h = 150, 150
        
        thumbnails = [_img_tag(p, max_w=max_w, max_h=max_h) for p in self.paths]

        rows_data = [
            ("File name", file_names), ("Printer", printers), ("Book", book_names),
            ("Page #", page_nums), ("Line #", line_nums), ("Char loc", char_locs),
            ("Image", thumbnails),
        ]

        def make_row(label, cells, extra_css='text-align:center;'):
            base_css = f'width:{cell_width}; max-width:{cell_width}; overflow-wrap:anywhere; word-break:break-all;'
            cell_html = ''.join(f'<td style="{base_css}{extra_css}">{c}</td>' for c in cells)
            return f'<tr><th scope="row" style="cursor:default;">{label}</th>{cell_html}</tr>'

        data_rows = '\n'.join(make_row(lbl, cells) for lbl, cells in rows_data)
        
        table_html = (
            f'<table id="{self.table_id}" {table_attrs}>\n'
            f'<thead>{header_row}</thead>\n'
            f'<tbody>{data_rows}</tbody>\n'
            f'</table>'
        )

        css = f"""
        <style>
            #{self.table_id} th, #{self.table_id} td {{ transition: background-color 0.3s ease; }}
            #{self.table_id} .selected {{ background-color: #d4edff !important; }}
            .export-button-{self.instance_id} {{
                margin-top: 15px; padding: 10px 15px; font-size: 14px; cursor: pointer;
                border: 1px solid #ccc; border-radius: 5px; background-color: #f0f0f0;
            }}
            .export-button-{self.instance_id}:hover {{ background-color: #e0e0e0; }}
        </style>
        """

        # --- THIS JAVASCRIPT BLOCK IS NOW CORRECT ---
        js = f"""
        <script>
        if (typeof window.toggleColumn !== 'function') {{
            window.toggleColumn = function(tableId, colIndex) {{
                const table = document.getElementById(tableId);
                const header = table.querySelector(`thead th:nth-child(${{colIndex + 2}})`);
                header.classList.toggle('selected');
                const rows = table.querySelectorAll('tbody tr');
                rows.forEach(row => {{
                    const cell = row.querySelector(`td:nth-child(${{colIndex + 2}})`);
                    if (cell) {{ cell.classList.toggle('selected'); }}
                }});
            }};
        }}

        if (typeof window.exportSelections !== 'function') {{
            window.exportSelections = function(tableId, instanceId) {{
                const table = document.getElementById(tableId);
                const selectedHeaders = table.querySelectorAll('thead th.selected');
                const selectedIndices = Array.from(selectedHeaders).map(th => {{
                    return Array.from(th.parentNode.children).indexOf(th) - 1;
                }});

                const payload = JSON.stringify({{ selected_indices: selectedIndices }});
                
                // ** THE FIX IS HERE **
                // We construct the command using standard JS concatenation to avoid f-string issues.
                const command = "_interactive_tables['" + instanceId + "']._handle_js_export(" + JSON.stringify(payload) + ")";

                console.log("Executing Python command:", command);
                if (typeof Jupyter !== 'undefined') {{
                    Jupyter.notebook.kernel.execute(command);
                }} else {{
                    console.error('Jupyter environment not found.');
                    alert('Error: Could not communicate with the Python kernel.');
                }}
            }};
        }}
        </script>
        """

        export_button = f"""
        <div>
            <button class="export-button-{self.instance_id}" onclick="exportSelections('{self.table_id}', '{self.instance_id}')">
                Export Selections
            </button>
        </div>
        """

        return f"{css}\n{js}\n{table_html}\n{export_button}"

    def display(self):
        html_content = self._generate_html()
        display(HTML(html_content))


# ---------------------------------------------------------------------------
# EXAMPLE
# ---------------------------------------------------------------------------
paths = [
    '/graft2/code/nvog/git/matching/char_images3/char_H_uc/'
    'reveringham_R30863_ctbtcml_2_fortysermons1685-012_page1rline8_char72_H_uc.tif',
    '/graft2/code/nvog/git/matching/char_images3/char_H_uc/'
    'reveringham_R30863_ctbtcml_2_fortysermons1685-056_page1rline37_char4_H_uc.tif',
    # … more paths …
]

# Create an instance of the interactive table and display it
# The annotations will be saved to 'local_annotations.json'
interactive_table = InteractiveAnnotationTable(paths, output_filename="local_annotations.json")
interactive_table.display()

In [ ]:
import html
import base64
import io
import json
import uuid
from pathlib import Path

from PIL import Image
from IPython.display import display, HTML
import ipywidgets as widgets # Ensure ipywidgets is imported

# This is a mock 'utils' module for demonstration.
# In your actual code, you would use your own 'utils.py' file.
# class MockUtils:
#     def get_printer_name(self, p): return "mock_printer"
#     def get_book_name(self, p): return "mock_book"
#     def get_page_num(self, p): return "1"
#     def get_line_num(self, p): return "1"
#     def get_char_loc(self, p): return "1"
# utils = MockUtils()


# ──────────────────────────────────────────────────────────────────
# 1. Low-level helper (Unchanged)
# ──────────────────────────────────────────────────────────────────
def _img_tag(path, *, max_w, max_h, resample=Image.Resampling.LANCZOS):
    try:
        with Image.open(path) as img:
            img.thumbnail((max_w, max_h), resample)
            buf = io.BytesIO()
            img.save(buf, format='PNG')
            b64 = base64.b64encode(buf.getvalue()).decode()
            style = (
                f'width:{max_w}px; height:{max_h}px; '
                'object-fit:contain; display:block; margin:auto;'
            )
            return f'<img src="data:image/png;base64,{b64}" style="{style}">'
    except Exception as exc:
        return f'<em>Preview error: {html.escape(str(exc))}</em>'


# ──────────────────────────────────────────────────────────────────
# 2. Corrected Class using ipywidgets Communication
# ──────────────────────────────────────────────────────────────────
class InteractiveAnnotationTable:
    """
    Generates an interactive HTML table.
    This version uses the ipywidgets comms channel and works in
    JupyterLab, VS Code, and classic Notebook.
    """
    def __init__(self, paths, output_filename="annotations.json"):
        if not paths:
            print("Warning: The 'paths' list is empty.")
            return

        self.paths = paths
        self.output_filename = output_filename
        self.table_id = f"annotation-table-{str(uuid.uuid4())}"
        
        # The widget that will contain our HTML and handle communication
        self._comm_widget = widgets.HTML()
        self._comm_widget.on_msg(self._handle_js_message)

    def _handle_js_message(self, widget, content, buffers):
        """Callback for when the frontend sends data."""
        event = content.get("event", "")
        if event == "export":
            data = content.get("payload", {})
            self._save_annotations(data)

    def _save_annotations(self, data):
        """Saves the selected file paths to a JSON file."""
        selected_indices = data.get("selected_indices", [])
        selected_paths = [self.paths[i] for i in selected_indices]

        with open(self.output_filename, 'w') as f:
            json.dump({
                "selected_paths": selected_paths,
                "selected_indices": selected_indices
            }, f, indent=4)

        # Display a confirmation message below the table
        confirmation = f"<p>✅ <b>Annotations saved!</b> {len(selected_paths)} items exported to '{self.output_filename}'</p>"
        self._status_widget.value = confirmation

    def _generate_html(self):
        """Generates the complete HTML string, including CSS and JavaScript."""
        comm_id = self._comm_widget.comm.comm_id
        n = len(self.paths)

        # ... (all the data gathering and table row generation logic is the same) ...
        # Simplified for brevity, assuming the full logic from previous steps
        file_names = [html.escape(Path(p).name) for p in self.paths]
        thumbnails = [f"Image for {Path(p).name}" for p in self.paths]
        rows_data = [("File name", file_names), ("Image", thumbnails)]

        header_cells = ''.join(
            f'<th style="cursor:pointer;" onclick="toggleColumn(\'{self.table_id}\', {i})">{i}</th>'
            for i in range(n)
        )
        header_row = f'<tr><th></th>{header_cells}</tr>'
        
        def make_row(label, cells):
            cell_html = ''.join(f'<td>{c}</td>' for c in cells)
            return f'<tr><th>{label}</th>{cell_html}</tr>'
        data_rows = '\n'.join(make_row(lbl, cells) for lbl, cells in rows_data)
        
        table_html = f'<table id="{self.table_id}" border="1" cellpadding="4">{header_row}{data_rows}</table>'

        css = f"""
        <style>
            #{self.table_id} th, #{self.table_id} td {{ transition: background-color 0.3s ease; }}
            #{self.table_id} .selected {{ background-color: #d4edff !important; }}
        </style>
        """

        js = f"""
        <script>
            // Ensure functions are defined only once
            if (!window.JupyterWidgetManager) {{
                window.JupyterWidgetManager = class {{
                    constructor(comm_id) {{
                        this.comm_id = comm_id;
                        this.comm = null;
                        this._get_comm();
                    }}

                    _get_comm() {{
                        if (window.jupyter && window.jupyter.widget) {{
                            // Classic Notebook
                            this.comm = window.jupyter.widget.comm_manager.comms['{comm_id}'];
                        }} else if (window.require) {{
                            // JupyterLab / VS Code
                            window.require(['@jupyter-widgets/base'], (widgets) => {{
                                widgets.ManagerBase.get_manager().then(manager => {{
                                    this.comm = manager.comms['{comm_id}'];
                                }});
                            }});
                        }}
                    }}

                    send(data) {{
                        if (this.comm) {{
                            this.comm.send(data);
                        }} else {{
                            console.error('Jupyter comms channel not available.');
                            alert('Error: Could not communicate with the Python kernel.');
                            // Try to re-establish comms
                            this._get_comm();
                        }}
                    }}
                }};
                window.commManager = new window.JupyterWidgetManager();
            }}
            
            if (typeof window.toggleColumn !== 'function') {{
                window.toggleColumn = (tableId, colIndex) => {{
                    // Selection logic remains the same
                    document.querySelectorAll(`#${{tableId}} tr`).forEach(row => {{
                        const cell = row.children[colIndex + 1];
                        if(cell) cell.classList.toggle('selected');
                    }});
                }};
            }}

            function exportSelections(tableId) {{
                const selectedIndices = Array.from(document.querySelectorAll(`#${{tableId}} thead th.selected`))
                    .map(th => Array.from(th.parentNode.children).indexOf(th) - 1);

                const payload = {{ event: 'export', payload: {{ selected_indices: selectedIndices }} }};
                window.commManager.send(payload);
            }}
        </script>
        """

        export_button = f"""
        <div style="margin-top: 15px;">
            <button onclick="exportSelections('{self.table_id}')">Export Selections</button>
        </div>
        """

        return f"{css}{js}{table_html}{export_button}"

    def display(self):
        """Renders the interactive table and status message widgets."""
        # This widget will hold the success/error messages
        self._status_widget = widgets.HTML()
        
        # Set the main widget's value to our generated HTML
        self._comm_widget.value = self._generate_html()

        # Display the interactive table and the status area below it
        display(widgets.VBox([self._comm_widget, self._status_widget]))

# ---------------------------------------------------------------------------
# EXAMPLE USAGE
# ---------------------------------------------------------------------------
# Use dummy paths since we don't have the real files/images
paths = [
    '/graft2/code/nvog/git/matching/char_images3/char_H_uc/'
    'reveringham_R30863_ctbtcml_2_fortysermons1685-012_page1rline8_char72_H_uc.tif',
    '/graft2/code/nvog/git/matching/char_images3/char_H_uc/'
    'reveringham_R30863_ctbtcml_2_fortysermons1685-056_page1rline37_char4_H_uc.tif',
    # … more paths …
]


# Create an instance of the interactive table and display it
interactive_table = InteractiveAnnotationTable(paths, output_filename="local_annotations.json")
interactive_table.display()

In [ ]:
data = load_matching_run('output/matching_results_may25/', 'query-manualvotes_paths_candidates-faiss_index_fortysermonsREDO_top_50_results')
# data = load_matching_run('output/matching_results_may25/', 'query-manualfortysermons_paths_candidates-faiss_index_fortysermonsREDO_top_50_results')


total_rows = sum([len(data[char]['queries']) for char in data.keys()])
print(f"Total rows: {total_rows}")
r = 0
max_r = 10
filter_char_class = True
filter_same_book_out = False
filter_same_page_out = True
min_sim = 0.0
topk = 10

for char in data.keys():
    print(f"Character {char} has {len(data[char]['queries'])} rows:")
    for i in range(len(data[char]['row_paths'])):
        if r > max_r:
            break
        # print(data[char]['row_paths'][i])
        query_char = utils.get_char(Path(data[char]['queries'][i]))
        query = data[char]['queries'][i]
        candidates = data[char]['candidates'][i]
        sims = data[char]['sims'][i]
        if filter_char_class:
            # only show candidates/sims that match the query character class
            valid_idxs = [n for n in range(len(candidates)) if utils.get_char(Path(candidates[n]).name) == query_char]
            candidates = [candidates[j] for j in valid_idxs]
            sims = [sims[j] for j in valid_idxs]
        if filter_same_book_out:
            valid_idxs = [n for n in range(len(candidates)) if utils.get_book_name(candidates[n]) != utils.get_book_name(query)]
            candidates = [candidates[j] for j in valid_idxs]
            sims = [sims[j] for j in valid_idxs]
        if filter_same_page_out:
            valid_idxs = [n for n in range(len(candidates)) if utils.get_page_num(candidates[n]) != utils.get_page_num(query)]
            candidates = [candidates[j] for j in valid_idxs]
            sims = [sims[j] for j in valid_idxs]
        if min_sim:
            valid_idxs = [n for n in range(len(candidates)) if sims[n] > min_sim]
            candidates = [candidates[j] for j in valid_idxs]
            sims = [sims[j] for j in valid_idxs]
        if len(candidates) == 0:
            # print(f"No candidates for query {query} with character {query_char}. Skipping.")
            continue
        print(f"Row: {r} / {len(data[char]['row_paths'])}")

        candidates = candidates[:topk]
        sims = sims[:topk]

        image_titles = get_image_titles(query, candidates, sims)

        paths = [query] + candidates
        html_table = paths_to_html_table_with_details(paths, cell_width='15ch')
        display(HTML(html_table))
        r += 1